In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
import numpy as np
from datasets import load_dataset

# Load the math dataset from Hugging Face
dataset = load_dataset("lighteval/MATH", split="train", trust_remote_code=True)
print(dataset)
print(dataset[0])
# randomly select a subset from the dataset
dataset = dataset.shuffle(seed=42)

Dataset({
    features: ['problem', 'level', 'type', 'solution'],
    num_rows: 7500
})
{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).', 'level': 'Level 5', 'type': 'Algebra', 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.'}


In [3]:
def select_questions_batch(dataset, indices):
    """Selects a batch of questions from the dataset by indices."""
    questions = [dataset[i]["problem"] for i in indices]
    solutions = [dataset[i]["solution"] for i in indices]
    return questions, solutions

In [2]:
import json
from vllm import LLM, SamplingParams

# Initialize the model
model_name = "meta-llama/Llama-3.2-3B-Instruct"
llm = LLM(model=model_name,tensor_parallel_size=2)
tokenizer = llm.get_tokenizer()

# note get strong error when using tensor_parallel_size=3 and os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,3"

INFO 12-26 15:59:09 config.py:350] This model supports multiple tasks: {'generate', 'embedding'}. Defaulting to 'generate'.
INFO 12-26 15:59:09 config.py:1020] Defaulting to use mp for distributed inference
WARNING 12-26 15:59:09 arg_utils.py:1013] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 12-26 15:59:09 config.py:1136] Chunked prefill is enabled with max_num_batched_tokens=512.
INFO 12-26 15:59:09 llm_engine.py:249] Initializing an LLM engine (v0.6.4.post1) with config: model='meta-llama/Llama-3.2-3B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=N

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(VllmWorkerProcess pid=3184711) INFO 12-26 15:59:27 model_runner.py:1077] Loading model weights took 3.0501 GB
INFO 12-26 15:59:27 model_runner.py:1077] Loading model weights took 3.0501 GB
(VllmWorkerProcess pid=3184711) INFO 12-26 15:59:28 worker.py:232] Memory profiling results: total_gpu_memory=47.43GiB initial_memory_usage=3.59GiB peak_torch_memory=3.10GiB memory_usage_post_profile=3.71GiB non_torch_memory=0.64GiB kv_cache_size=38.94GiB gpu_memory_utilization=0.90
INFO 12-26 15:59:28 worker.py:232] Memory profiling results: total_gpu_memory=47.43GiB initial_memory_usage=3.59GiB peak_torch_memory=4.23GiB memory_usage_post_profile=3.73GiB non_torch_memory=0.66GiB kv_cache_size=37.80GiB gpu_memory_utilization=0.90
INFO 12-26 15:59:28 distributed_gpu_executor.py:57] # GPU blocks: 44231, # CPU blocks: 4681
INFO 12-26 15:59:28 distributed_gpu_executor.py:61] Maximum concurrency for 131072 tokens per request: 5.40x
INFO 12-26 15:59:34 model_runner.py:1400] Capturing cudagraphs for decodi

: 

In [5]:
def return_entropy(logprobs):
    # Calculate entropy for each token, return a list of entropies
    entropies = []
    num_tokens = len(logprobs) # number of tokens
    for i in range(num_tokens):
        temp = []
        for token_id in logprobs[i].keys():
            temp.append(logprobs[i][token_id].logprob)
        temp = np.array(temp)
        entropy = -1 * np.sum(np.exp(temp) * temp)
        entropies.append(entropy)
    return entropies

# print(outputs_step1[0].outputs[0].logprobs[0])
# result = return_entropy(outputs_step1[0].outputs[0].logprobs) # first prompt, first generation
# print(len(result))
# print(result)

In [6]:
def return_probs(logprobs):
    # Calculate entropy for each token, return a list of probs
    probs = []
    num_tokens = len(logprobs) # number of tokens
    for i in range(num_tokens):
        token_id = next(iter(logprobs[i]))
        probs.append(logprobs[i][token_id].logprob)
    return probs
# print(outputs_step1[0].outputs[0].logprobs[0])
# result = return_probs(outputs_step1[0].outputs[0].logprobs) # first prompt, first generation
# print(len(result))
# print(result)

In [ ]:
# Define parameters

n1 = 8  # Number of results for each prompt in Step 1
n2 = 32  # Number of free generations for each result in Step 2
max_tokens_step1 = 256
max_tokens_step2 = 1024
batch_size = 64 

# Initialize storage for results
all_results_step1 = []
num_batches = (len(dataset) + batch_size - 1) // batch_size
# Process dataset in batches
for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(dataset))
    indices = list(range(start_idx, end_idx))

    prompts, solutions = select_questions_batch(dataset, indices)
    system_prompt = "You are a helpful assistant for math problem-solving. At the end of the solution, provide the final answer in the format: \\boxed{answer}. Now solve the following problem: "
    prompts = [system_prompt + prompt for prompt in prompts]

    # Step 1: Generate limited responses with logprobs
    sampling_params_step1 = SamplingParams(
        temperature=0.9,
        max_tokens=max_tokens_step1,
        n=n1,
        stop=["Step 3"],
        logprobs=20,
        stop_token_ids=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
        skip_special_tokens = False,
        include_stop_str_in_output = True
    )

    outputs_step1 = llm.generate(prompts, sampling_params_step1)

    # Store results for Step 1
    for output in outputs_step1:
        for gen_output in output.outputs:
            all_results_step1.append({
                "original_prompt": output.prompt,
                "original_prompt_tokens": output.prompt_token_ids,
                "generated_text": tokenizer.decode(gen_output.token_ids, skip_special_tokens=False),
                "generated_tokens": gen_output.token_ids,
                "entropy": return_entropy(gen_output.logprobs),
                "logprobs": return_probs(gen_output.logprobs),  
            })

print("Step 1 complete! ")    

# Step 2: Free-generation for each result of Step 1
all_results_step2 = []
sampling_params_step2 = SamplingParams(
    temperature=0.9,
    max_tokens=max_tokens_step2,
    n=n2,
    # seed=42,  # wrong, this will always return the same result even when n > 1
    stop_token_ids=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
    logprobs=20,
)

combined_tokens_batches = [step1_result["original_prompt_tokens"][1:] + list(step1_result["generated_tokens"]) for step1_result in all_results_step1] # remove the BOS token
combined_text_batches = tokenizer.batch_decode(combined_tokens_batches, skip_special_tokens=False)
num_batches = (len(combined_text_batches) + batch_size - 1) // batch_size
for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(combined_text_batches))
    indices = list(range(start_idx, end_idx))

    combined_text_batch = combined_text_batches[start_idx:end_idx]
    outputs_step2 = llm.generate(combined_text_batch, sampling_params_step2)
    # Store results for Step 2
    for output in outputs_step2:
        for gen_output in output.outputs:
            all_results_step2.append({
                "first_step_text": output.prompt,
                "first_step_tokens": output.prompt_token_ids,
                "generated_text": gen_output.text,
                "generated_tokens": gen_output.token_ids,
                "entropy": return_entropy(gen_output.logprobs),
                "logprobs": return_probs(gen_output.logprobs),  
            })


print("Batch inference complete! ")


Processed prompts:  12%|█▎        | 12/96 [00:05<00:35,  2.34it/s, est. speed input: 230.34 toks/s, output: 3009.47 toks/s]


Step 1 complete! 


Processed prompts:   2%|▏         | 36/2048 [03:02<2:09:04,  3.85s/it, est. speed input: 54.69 toks/s, output: 2364.01 toks/s]

(VllmWorkerProcess pid=1632002) WARNING 12-20 19:02:05 shm_broadcast.py:391] No available block found in 60 second. 


In [8]:
# save the results to a json file
with open("step1_test.jsonl", "w") as f:
    for result in all_results_step1:
        f.write(json.dumps(result) + "\n")
        
with open("step2_test.jsonl", "w") as f:
    for result in all_results_step2:
        f.write(json.dumps(result) + "\n")

In [12]:
import json

results = []
with open("step2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        results.append(json.loads(line))

In [13]:
results[0]

{'first_step_text': 'You are a helpful assistant for math problem-solving. At the end of the solution, provide the final answer in the format: \\boxed{answer}. Now solve the following problem: What is the number of units in the distance between $(2,5)$ and $(-6,-1)$? \n## Step 1: Identify the coordinates of the two points\nThe coordinates of the two points are given as $(2,5)$ and $(-6,-1)$.\n\n## Step 2: Recall the distance formula\nThe distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ can be found using the distance formula: $distance = \\sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$.\n\n## Step 3',
 'first_step_tokens': [128000,
  2675,
  527,
  264,
  11190,
  18328,
  369,
  7033,
  3575,
  99246,
  13,
  2468,
  279,
  842,
  315,
  279,
  6425,
  11,
  3493,
  279,
  1620,
  4320,
  304,
  279,
  3645,
  25,
  1144,
  80175,
  90,
  9399,
  7966,
  4800,
  11886,
  279,
  2768,
  3575,
  25,
  3639,
  374,
  279,
  1396,
  315,
  8316,
  304,
  279,
  6138,
  1990,
  5035,
  17,
  

In [14]:
for i in range(5):
    print(all_results[i]['first_step_text'])
    print('-' * 50)
    print(all_results[i]['generated_text'])
    print('-' * 50)
    print(len(all_results[i]['entropy']))
    print(len(all_results[i]['logprobs']))
    print('-' * 50)
    print(len(all_results[i]['generated_tokens']))

You are a helpful assistant for math problem-solving. At the end of the solution, provide the final answer in the format: \boxed{answer}. Now solve the following problem: What is the number of units in the distance between $(2,5)$ and $(-6,-1)$? 
## Step 1: Identify the coordinates of the two points
The two points given are $(2,5)$ and $(-6,-1)$.

## Step 2: Recall the distance formula
The distance formula to find the distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ is $\sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}$.

## Step 3
--------------------------------------------------
: Apply the distance formula to the given points
Substituting the coordinates into the formula, we get $\sqrt{((-6)-2)^2 + ((-1)-5)^2}$.

## Step 4: Perform the arithmetic operations inside the square root
Simplifying inside the square root gives $\sqrt{(-8)^2 + (-6)^2}$.

## Step 5: Calculate the squared values
This becomes $\sqrt{64 + 36}$.

## Step 6: Sum the values inside the square root
This equals $\sqrt{100}$.


In [71]:
if hasattr(llm, "__dict__"):
    for key, value in llm.__dict__.items():
        print(f"{key}: {value}")
else:
    print("The object does not have a __dict__ attribute.")



engine_class: <class 'vllm.engine.llm_engine.LLMEngine'>
llm_engine: <vllm.engine.llm_engine.LLMEngine object at 0x7fe69a5ced00>
request_counter: <vllm.utils.Counter object at 0x7fe69a5ce370>
